# Phase 6 automated tests: `notebooks/02_silver_transformation.py`

| Suite | Scope |
|---|---|
| `TestSilverUnitLogic` | offline, plain PySpark (ANSI mode ON, like Databricks Serverless) |
| `TestSilverChildTables` | projections, keys, grain |
| `TestSilverDeltaIntegration` | needs Delta JARs; skipped automatically when they cannot be loaded |

Run all suites from the last cell. Set `PROJECT_ROOT` if this notebook is not in `tests/`.

In [0]:
# Install dependencies (run once) — pyspark / delta-spark ship with the runtime and must not be pinned
%pip install -q pytest ipytest

## Setup: imports, paths and the module under test

In [0]:
import hashlib, hmac, importlib.util, json, os, sys, textwrap
from datetime import datetime, date

import pytest

# Notebook has no __file__: assume this notebook lives in tests/ (override with PROJECT_ROOT)
ROOT = os.environ.get("PROJECT_ROOT") or os.path.abspath(os.path.join(os.getcwd(), ".."))
if not ROOT.startswith("/Workspace"):            # kernel cwd may report workspace-relative paths
    ROOT = "/Workspace" + ROOT                   # filesystem access needs the /Workspace prefix
sys.path.insert(0, os.path.join(ROOT, "notebooks"))  # makes 00_config / 00_audit_logger importable

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType, BooleanType,
                               DateType, TimestampType)

# The module under test is a Databricks notebook, not a .py file: use a .py export when present,
# otherwise load 02_silver_transformation.ipynb by executing its code cells (dedented per cell).
_silver_py = os.path.join(ROOT, "notebooks", "02_silver_transformation.py")
_silver_ipynb = os.path.join(ROOT, "notebooks", "02_silver_transformation.ipynb")
if os.path.isfile(_silver_py):
    _spec = importlib.util.spec_from_file_location("silver", _silver_py)
    silver = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(silver)
else:
    with open(_silver_ipynb, "r", encoding="utf-8") as fh:
        _cells = json.load(fh)["cells"]
    _code = ["".join(c["source"]) if isinstance(c["source"], list) else c["source"]
             for c in _cells if c.get("cell_type") == "code"]
    silver = importlib.util.module_from_spec(importlib.util.spec_from_loader("silver", None))
    silver.__file__ = _silver_ipynb
    exec(compile("\n\n".join(textwrap.dedent(c) for c in _code), _silver_ipynb, "exec"), silver.__dict__)

PEPPER = "unit-test-pepper"
LOAD_TS = datetime(2026, 10, 20, 12, 0, 0)
SCHEMA_DIR = os.path.join(ROOT, "data", "schemas")

## Bronze / reference schemas

In [0]:
BRONZE_SCHEMA = StructType([
    StructField("measurement_uid", StringType()), StructField("report_id", StringType()),
    StructField("input", StringType()), StructField("test_name", StringType()),
    StructField("measurement_start_time", StringType()), StructField("probe_asn", StringType()),
    StructField("probe_cc", StringType()), StructField("probe_ip", StringType()),
    StructField("probe_network_name", StringType()), StructField("resolver_asn", StringType()),
    StructField("resolver_ip", StringType()), StructField("test_runtime", DoubleType()),
    StructField("test_keys", StringType()), StructField("api_anomaly", BooleanType()),
    StructField("api_confirmed", BooleanType()), StructField("measurement_date", DateType()),
    StructField("load_timestamp", TimestampType()),
])
REF_SCHEMA = StructType([StructField(n, StringType()) for n in
                         ("url", "category_code", "category_description", "country_code", "taxonomy_version")])

## Fixtures and helpers

In [0]:
@pytest.fixture(scope="module")
def spark():
    s = SparkSession.getActiveSession()
    if s is None or "connect" not in type(s).__module__:
        # classic compute: dedicated throwaway local session (original behaviour)
        s = (SparkSession.builder.master("local[2]").appName("silver-phase6")
             .config("spark.sql.shuffle.partitions", "2").config("spark.sql.session.timeZone", "UTC")
             .config("spark.sql.ansi.enabled", "true").config("spark.ui.enabled", "false").getOrCreate())
        yield s
        s.stop()
    else:
        # Spark Connect (serverless): .master() is forbidden (CANNOT_CONFIGURE_SPARK_CONNECT_MASTER),
        # so reuse the notebook's remote session and set the SQL configs on it; never stop it.
        for k, v in (("spark.sql.shuffle.partitions", "2"), ("spark.sql.session.timeZone", "UTC"),
                     ("spark.sql.ansi.enabled", "true")):
            s.conf.set(k, v)
        yield s

In [0]:
def row(uid=None, report_id="R1", inp="https://example.org/", test="web_connectivity", start="2026-09-27 10:00:00",
        asn="AS9541", cc="PK", pip="127.0.0.1", resolver_ip="202.163.69.18", rasn="AS9541", runtime=1.5,
        tk=None, api_anomaly=False, api_confirmed=False, bronze_ts=datetime(2026, 9, 27, 11, 0, 0), name="ISP"):
    tk_json = tk if isinstance(tk, str) else json.dumps(tk if tk is not None else {"blocking": False, "accessible": True})
    return (uid, report_id, inp, test, start, asn, cc, pip, name, rasn, resolver_ip, runtime, tk_json,
            api_anomaly, api_confirmed, date(2026, 9, 27), bronze_ts)


def taxonomy(spark, version="commit-test"):
    rows = [("https://www.bbc.com/news/persian", "NEWS", "News Media", "GLOBAL", version),
            ("https://twitter.com/", "COMM", "Communication Tools", "GLOBAL", version),
            ("https://twitter.com/politics", "POLR", "Political Criticism", "GLOBAL", version),
            ("https://dual.example/", "FILE", "File Sharing", "GLOBAL", version),
            ("https://dual.example/", "CULTR", "Culture", "PK", version),
            ("http://rights.example/", "HUMR", "Human Rights", "GLOBAL", version),
            ("https://faith.example/", "REL", "Religion", "GLOBAL", version)]
    return spark.createDataFrame(rows, REF_SCHEMA)


def run(spark, rows, ref=None):
    df = spark.createDataFrame(rows, BRONZE_SCHEMA)
    return silver.transform_bronze_to_silver(df, ref if ref is not None else taxonomy(spark), PEPPER, "B1", LOAD_TS)


def parent(out):
    return {r["report_id"]: r.asDict() for r in out["parent"].collect()}

## `TestSilverUnitLogic`

In [0]:
class TestSilverUnitLogic:

    # ---- contracts / validator (checklist 4, 22) ----
    def test_schema_contracts_match_json_and_validator_checks_types(self, spark):
        out = run(spark, [row()])
        pairs = [("parent", "silver_conformed_schema.json"), ("dns", "silver_dns_experiments_schema.json"),
                 ("tcp", "silver_tcp_experiments_schema.json"), ("app", "silver_app_anomaly_schema.json")]
        for key, fn in pairs:
            ok, errors = silver.validate_silver_schema_contract(out[key], os.path.join(SCHEMA_DIR, fn))
            assert ok, (fn, errors)
        # a type change must be caught (names-only validators would pass this)
        bad = out["parent"].withColumn("probe_asn", F.col("probe_asn").cast("string"))
        ok, errors = silver.validate_silver_schema_contract(bad, os.path.join(SCHEMA_DIR, "silver_conformed_schema.json"))
        assert not ok and any("probe_asn" in e and "Type mismatch" in e for e in errors)
        extra = out["parent"].withColumn("surprise", F.lit(1))
        assert any("Unexpected extra column" in e for e in
                   silver.validate_silver_schema_contract(extra, os.path.join(SCHEMA_DIR, "silver_conformed_schema.json"))[1])

    def test_load_timestamp_on_every_silver_table(self, spark):
        out = run(spark, [row(test="whatsapp", tk={"whatsapp_web_status": "ok"})])
        for key in ("parent", "dns", "tcp", "app"):
            assert "load_timestamp" in out[key].columns

    # ---- identity (checklist 7, 8, 9) ----
    def test_measurement_id_prefers_uid_else_deterministic_hash(self, spark):
        g = parent(run(spark, [row(uid="20260927T100000Z_webconn_PK_1", report_id="U"), row(report_id="H")]))
        assert g["U"]["measurement_id"] == "20260927T100000Z_webconn_PK_1"
        parts = ["PK", "2026-09-27", "H", "2026-09-27 10:00:00", "web_connectivity", "https://example.org/"]
        expect = hashlib.sha256("|".join(f"V{len(x)}:{x}" for x in parts).encode()).hexdigest()
        assert g["H"]["measurement_id"] == expect

    def test_same_timestamp_different_natural_keys_stay_distinct(self, spark):
        rows = [row(report_id="A"), row(report_id="B"), row(report_id="A", inp="https://other.org/"),
                row(report_id="A", test="telegram", tk={})]
        assert run(spark, rows)["parent"].select("measurement_id").distinct().count() == 4

    def test_key_components_cannot_collide(self, spark):
        rows = [row(report_id="a|b", inp="c"), row(report_id="a", inp="b|c"),          # delimiter shifting
                row(report_id="N1", inp=None), row(report_id="N1", inp="")]              # NULL vs empty
        assert run(spark, rows)["parent"].select("measurement_id").distinct().count() == 4

    def test_duplicates_collapse_before_merge_newest_wins(self, spark):
        older, newer = datetime(2026, 9, 27, 11), datetime(2026, 9, 27, 12)
        out = run(spark, [row(bronze_ts=older, runtime=1.0), row(bronze_ts=newer, runtime=2.0)])
        rows = out["parent"].collect()
        assert len(rows) == 1 and rows[0]["test_runtime_seconds"] == 2.0

    # ---- privacy (checklist 11, 12) ----
    def test_no_forbidden_columns_or_raw_ips_anywhere(self, spark):
        out = run(spark, [row(pip="203.0.113.77", resolver_ip="202.163.69.18",
                              tk={"queries": [{"failure": None, "query_type": "A", "hostname": "example.org",
                                               "resolver_address": "9.9.9.9:53",
                                               "answers": [{"answer_type": "A", "ipv4": "93.184.216.34"}]}],
                                  "client_resolver": "202.163.69.18"})])
        for key in ("parent", "dns", "tcp", "app"):
            assert not (set(out[key].columns) & silver.FORBIDDEN_SILVER_COLUMNS), key
        for schema in (silver.SILVER_SCHEMA, silver.SILVER_DNS_SCHEMA, silver.SILVER_TCP_SCHEMA, silver.SILVER_APP_SCHEMA):
            assert not ({f.name for f in schema.fields} & silver.FORBIDDEN_SILVER_COLUMNS)
        blob = json.dumps([str(v) for key in ("parent", "dns") for r in out[key].collect() for v in r.asDict().values()])
        for secret in ("203.0.113.77", "202.163.69.18", "9.9.9.9"):
            assert secret not in blob

    def test_resolver_hash_is_hmac_and_salt_sensitive(self, spark):
        g = parent(run(spark, [row(report_id="X")]))
        assert g["X"]["hashed_resolver_ip"] == hmac.new(PEPPER.encode(), b"202.163.69.18", hashlib.sha256).hexdigest()
        other = silver.hash_resolver_ip(spark.createDataFrame([("202.163.69.18",)], "resolver_ip string"), "other-pepper")
        assert other.first()["hashed_resolver_ip"] != g["X"]["hashed_resolver_ip"]

    def test_empty_or_loopback_resolver_gives_null_hash(self, spark):
        g = parent(run(spark, [row(report_id="L", resolver_ip="127.0.0.1"), row(report_id="E", resolver_ip=""),
                               row(report_id="N", resolver_ip=None)]))
        assert all(g[k]["hashed_resolver_ip"] is None for k in ("L", "E", "N"))

    def test_missing_secret_fails_closed(self, monkeypatch):
        monkeypatch.delenv("PEPPER_HASH_SALT", raising=False)
        monkeypatch.delenv("PANOPTICON_SALT", raising=False)
        with pytest.raises(ValueError):
            silver.resolve_pepper_salt(None)
        with pytest.raises(ValueError):
            silver.make_hmac_udf("")
        assert silver.resolve_pepper_salt("from-arg") == "from-arg"
        monkeypatch.setenv("PEPPER_HASH_SALT", "from-env")
        assert silver.resolve_pepper_salt(None) == "from-env"

    def test_subnet_masking_and_url_sanitising(self, spark):
        g = parent(run(spark, [row(report_id="A", pip="203.0.113.77", inp="https://u:pw@example.org/p?token=abc&q=1"),
                               row(report_id="B", pip="127.0.0.1"), row(report_id="C", pip="2001:db8:abcd:12:1:2:3:4")]))
        assert g["A"]["masked_probe_subnet"] == "203.0.113.0/24"
        assert g["A"]["target_url"] == "https://example.org/p?token=REDACTED&q=1"
        assert g["B"]["masked_probe_subnet"] is None                     # loopback is uninformative
        assert g["C"]["masked_probe_subnet"] == "2001:db8:abcd::/48"

    def test_asn_fallbacks_never_null(self, spark):
        g = parent(run(spark, [row(report_id="A", asn="as9541", rasn="AS0"), row(report_id="B", asn="", rasn=None),
                               row(report_id="C", asn="garbage", rasn="AS99999999999999999999")]))
        assert (g["A"]["probe_asn"], g["A"]["resolver_asn"]) == (9541, 0)
        assert (g["B"]["probe_asn"], g["B"]["resolver_asn"]) == (0, 0)
        assert (g["C"]["probe_asn"], g["C"]["resolver_asn"]) == (0, 0)

    def test_quarantine_rules(self, spark):
        out = run(spark, [row(report_id=None), row(report_id="BT", start="27/09/2026 10:00"),
                          row(report_id="BC", cc="Pakistan"), row(report_id="BN", test=""), row(report_id="OK")])
        assert [r["report_id"] for r in out["parent"].collect()] == ["OK"]
        reasons = sorted(r["_dq_reason"] for r in out["invalid"].collect())
        assert reasons == sorted(["MISSING_REPORT_ID", "UNPARSEABLE_MEASUREMENT_START_TIME",
                                  "INVALID_PROBE_CC", "MISSING_TEST_NAME"])
        assert all("127.0.0.1" not in (r["raw_record"] or "") for r in out["invalid"].collect())

    # ---- URL normalisation + enrichment (checklist 15, 16) ----
    def test_exact_url_beats_domain_fallback(self, spark):
        g = parent(run(spark, [row(report_id="E", inp="https://bbc.com/news/persian"),                 # scheme/www differ
                               row(report_id="D", inp="https://www.bbc.com/other/page")]))
        assert (g["E"]["category_match_type"], g["E"]["content_category"]) == ("EXACT_URL", "NEWS")
        assert g["D"]["category_match_type"] == "DOMAIN_FALLBACK"   # a different path on the same host only matches at domain level
        assert g["D"]["content_category"] == "NEWS"

    def test_subdomain_domain_fallback_and_priority(self, spark):
        g = parent(run(spark, [row(report_id="S", inp="https://subdomain.twitter.com/random_page")]))
        # twitter.com has COMM and POLR entries: political category has higher priority
        assert g["S"]["category_match_type"] == "DOMAIN_FALLBACK" and g["S"]["content_category"] == "POLR"

    def test_unmatched_and_null_inputs_never_dropped_and_versioned(self, spark):
        g = parent(run(spark, [row(report_id="U", inp="https://internal-untracked-site.local"),
                               row(report_id="N", inp=None, test="telegram", tk={}),
                               row(report_id="NI", inp="NO_INPUT", test="signal", tk={})]))
        for k in ("U", "N", "NI"):
            assert (g[k]["category_match_type"], g[k]["content_category"]) == ("UNMATCHED", "MISC")
            assert g[k]["taxonomy_version"] == "commit-test"          # snapshot used, never NULL/UNKNOWN
            assert g[k]["human_rights_risk_tier"] == "LOW"

    def test_probe_country_entry_beats_global_and_tiers(self, spark):
        g = parent(run(spark, [row(report_id="P", inp="https://dual.example/", cc="PK"),
                               row(report_id="I", inp="https://dual.example/", cc="IR"),
                               row(report_id="H", inp="http://rights.example/"), row(report_id="R", inp="https://faith.example/")]))
        assert g["P"]["content_category"] == "CULTR" and g["I"]["content_category"] == "FILE"
        assert g["H"]["human_rights_risk_tier"] == "HIGH" and g["R"]["human_rights_risk_tier"] == "MEDIUM"
        assert g["I"]["human_rights_risk_tier"] == "MEDIUM" and g["P"]["human_rights_risk_tier"] == "LOW"

    def test_enrichment_does_not_multiply_rows(self, spark):
        rows = [row(report_id=f"M{i}", inp="https://twitter.com/x") for i in range(10)]
        assert run(spark, rows)["parent"].count() == 10

    def test_empty_taxonomy_degrades_to_unmatched(self, spark):
        empty = spark.createDataFrame([], REF_SCHEMA)
        g = parent(run(spark, [row(report_id="Z")], ref=empty))
        assert g["Z"]["category_match_type"] == "UNMATCHED" and g["Z"]["taxonomy_version"] == "UNKNOWN"

    # ---- epistemic classification (checklist 13, 14) ----
    def test_https_blind_spot_resolved(self, spark):
        tk = {"blocking": False, "tls_handshakes": [{"failure": "connection_reset"}]}
        r = parent(run(spark, [row(report_id="T", tk=tk)]))["T"]
        assert (r["tls_anomaly_flag"], r["tampering_vector"], r["classification_status"]) == (1, "TLS_DROP", "HEURISTIC_ANOMALY")
        assert r["is_anomaly"] is True

    def test_benign_is_not_unclassified(self, spark):
        r = parent(run(spark, [row(report_id="B", tk={"blocking": False, "accessible": True})]))["B"]
        assert (r["is_anomaly"], r["classification_status"], r["tampering_vector"]) == (False, "BENIGN", "BENIGN")
        assert (r["dns_anomaly_flag"], r["tcp_anomaly_flag"], r["tls_anomaly_flag"], r["http_anomaly_flag"]) == (0, 0, 0, 0)

    def test_confirmed_blockpage(self, spark):
        r = parent(run(spark, [row(report_id="C", api_confirmed=True, api_anomaly=True, tk={"blocking": "http-diff"})]))["C"]
        assert (r["classification_status"], r["tampering_vector"], r["ooni_confirmed"]) == ("CONFIRMED_BLOCKED", "HTTP_BLOCK", True)
        d = parent(run(spark, [row(report_id="D", api_confirmed=True, tk={"blocking": "dns"})]))["D"]
        assert d["classification_status"] == "CONFIRMED_BLOCKED" and d["tampering_vector"] == "DNS_TAMPERING"

    def test_hierarchical_precedence(self, spark):
        tk = {"dns_experiment_failure": "dns_nxdomain_error", "requests": [{"failure": None, "response": {"code": 403}}],
              "tls_handshakes": [{"failure": "ssl_error"}], "tcp_connect": [{"ip": "1.2.3.4", "port": 443, "status": {"failure": "connection_reset", "success": False}}]}
        r = parent(run(spark, [row(report_id="P", tk=tk)]))["P"]
        assert r["tampering_vector"] == "DNS_TAMPERING"
        assert (r["dns_anomaly_flag"], r["tcp_anomaly_flag"], r["tls_anomaly_flag"], r["http_anomaly_flag"]) == (1, 1, 1, 1)
        tk2 = {"tcp_connect": [{"ip": "1.2.3.4", "port": 443, "status": {"failure": "connection_reset"}}], "tls_handshakes": [{"failure": "x"}]}
        assert parent(run(spark, [row(report_id="Q", tk=tk2)]))["Q"]["tampering_vector"] == "TCP_RESET"
        tk3 = {"requests": [{"failure": None, "response": {"code": 451}}]}
        assert parent(run(spark, [row(report_id="R", tk=tk3)]))["R"]["tampering_vector"] == "HTTP_BLOCK"

    @pytest.mark.parametrize("tk,test,api,vector,flags", [
        ({"blocking": False}, "web_connectivity", False, "BENIGN", (0, 0, 0, 0)),
        ({"blocking": "dns"}, "web_connectivity", False, "DNS_TAMPERING", (1, 0, 0, 0)),
        ({"blocking": "tcp_ip"}, "web_connectivity", False, "TCP_RESET", (0, 1, 0, 0)),
        ({"tls_handshakes": [{"failure": "generic_timeout_error"}]}, "web_connectivity", False, "TLS_DROP", (0, 0, 1, 0)),
        ({"http_experiment_failure": "connection_refused"}, "web_connectivity", False, "HTTP_BLOCK", (0, 0, 0, 1)),
        ({"telegram_tcp_blocking": True}, "telegram", False, "TCP_RESET", (0, 1, 0, 0)),
        ({"telegram_web_status": "blocked"}, "telegram", False, "HTTP_BLOCK", (0, 0, 0, 1)),
        ({"whatsapp_endpoints_dns_inconsistent": ["x"]}, "whatsapp", False, "DNS_TAMPERING", (1, 0, 0, 0)),
        ({"whatsapp_endpoints_dns_inconsistent": []}, "whatsapp", False, "BENIGN", (0, 0, 0, 0)),
        ({"signal_backend_status": "blocked"}, "signal", False, "TLS_DROP", (0, 0, 1, 0)),
        ({"facebook_tcp_blocking": True}, "facebook_messenger", False, "TCP_RESET", (0, 1, 0, 0)),
        ({"x": 1}, "stunreachability", True, "ANOMALY_UNCLASSIFIED", (0, 0, 0, 0)),
        ({"x": 1}, "stunreachability", False, "BENIGN", (0, 0, 0, 0)),
    ])
    def test_vector_flag_consistency(self, spark, tk, test, api, vector, flags):
        r = parent(run(spark, [row(report_id="V", test=test, tk=tk, api_anomaly=api)]))["V"]
        assert r["tampering_vector"] == vector
        assert (r["dns_anomaly_flag"], r["tcp_anomaly_flag"], r["tls_anomaly_flag"], r["http_anomaly_flag"]) == flags
        assert r["is_anomaly"] == (vector not in ("BENIGN", "UNCLASSIFIED"))

    def test_unknown_or_malformed_never_counts_as_anomaly(self, spark):
        rows = [row(report_id="M", tk="{not json", api_anomaly=None, api_confirmed=None),        # no ground truth, no payload
                row(report_id="K", tk="{not json", api_anomaly=False),                           # ground truth says benign
                row(report_id="N", tk={"x": 1}, api_anomaly=None, api_confirmed=None)]           # parseable, no evidence
        g = parent(run(spark, rows))
        assert (g["M"]["classification_status"], g["M"]["tampering_vector"], g["M"]["is_anomaly"]) == ("UNCLASSIFIED", "UNCLASSIFIED", False)
        assert g["K"]["classification_status"] == "BENIGN" and g["K"]["is_anomaly"] is False
        assert g["N"]["classification_status"] == "BENIGN" and g["N"]["is_anomaly"] is False

    def test_null_ground_truth_does_not_poison_classification(self, spark):
        r = parent(run(spark, [row(report_id="X", api_anomaly=None, api_confirmed=None,
                                   tk={"dns_experiment_failure": "dns_nxdomain_error"})]))["X"]
        assert r["classification_status"] == "HEURISTIC_ANOMALY" and r["ooni_anomaly"] is False and r["ooni_confirmed"] is False

    def test_enumerations_are_closed(self, spark):
        out = run(spark, [row(report_id="A"), row(report_id="B", tk={"blocking": "dns"}, api_anomaly=True),
                          row(report_id="C", api_confirmed=True)])
        for r in out["parent"].collect():
            assert r["tampering_vector"] in silver.TAMPERING_VECTORS
            assert r["classification_status"] in silver.CLASSIFICATION_STATUSES
            assert r["category_match_type"] in silver.CATEGORY_MATCH_TYPES
            assert r["human_rights_risk_tier"] in ("HIGH", "MEDIUM", "LOW")

    def test_runtime_is_named_and_kept_as_seconds(self, spark):
        out = run(spark, [row(report_id="R", runtime=4.25), row(report_id="S", runtime=None)])
        assert "response_latency_ms" not in out["parent"].columns
        g = parent(out)
        assert g["R"]["test_runtime_seconds"] == 4.25 and g["S"]["test_runtime_seconds"] is None

    def test_base64_body_and_structural_surprises_do_not_break_classification(self, spark):
        tk = {"requests": [{"failure": None, "response": {"code": 200, "body": {"data": "R0lG", "format": "base64"}}}],
              "queries": {"unexpected": "object-not-array"},                      # structural conflict
              "tcp_connect": [{"ip": "1.2.3.4", "port": "443", "status": {"failure": "connection_reset"}}]}
        r = parent(run(spark, [row(report_id="Z", tk=tk)]))["Z"]
        assert r["tcp_anomaly_flag"] == 1 and r["tampering_vector"] == "TCP_RESET"   # tcp still parsed independently

    def test_timestamp_edge_cases_never_crash_under_ansi(self, spark):
        rows = [row(report_id="ISO", start="2026-09-27T10:00:00Z"), row(report_id="EMPTY", start=""),
                row(report_id="NA", start="N/A"), row(report_id="GOOD")]
        out = run(spark, rows)
        assert sorted(r["report_id"] for r in out["parent"].collect()) == ["GOOD", "ISO"]
        assert out["invalid"].count() == 2

    # ---- drift guard ----
    def test_drift_extra_missing_and_type_changes(self, spark):
        df = spark.createDataFrame([row()], BRONZE_SCHEMA).withColumn("brand_new_field", F.lit("x"))
        out, rep = silver.detect_and_absorb_schema_drift(df)
        assert rep["unexpected_columns"] == ["brand_new_field"] and out.columns == list(silver.BRONZE_INPUT_CONTRACT)
        changed = (spark.createDataFrame([row(report_id="A"), row(report_id="B")], BRONZE_SCHEMA)
                   .withColumn("test_runtime", F.when(F.col("report_id") == "A", F.lit("1.25")).otherwise(F.lit("oops"))))
        out2, rep2 = silver.detect_and_absorb_schema_drift(changed)
        got = {r["report_id"]: r["test_runtime"] for r in out2.collect()}
        assert got == {"A": 1.25, "B": None} and "test_runtime" in rep2["type_changes"]
        with pytest.raises(silver.SchemaDriftError, match="Re-run Bronze"):
            silver.detect_and_absorb_schema_drift(spark.createDataFrame([row()], BRONZE_SCHEMA).drop("api_anomaly"))

    def test_parameter_windows(self):
        p = silver.resolve_silver_parameters(batch_date="2026-09-28", load_type="incremental", lookback_days=3)
        assert (p["window_start"], p["window_end"]) == ("2026-09-25", "2026-09-28")
        p = silver.resolve_silver_parameters(batch_date="2026-09-22", load_type="backfill", lookback_days=0)
        assert (p["window_start"], p["window_end"]) == ("2026-09-22", "2026-09-22")
        p = silver.resolve_silver_parameters(load_type="backfill", backfill_start="2026-09-10", backfill_end="2026-09-14")
        assert (p["window_start"], p["window_end"]) == ("2026-09-10", "2026-09-14")
        assert silver.resolve_silver_parameters(load_type="full")["window_start"] is None
        with pytest.raises(ValueError):
            silver.resolve_silver_parameters(load_type="bogus")

## `TestSilverChildTables`

In [0]:
class TestSilverChildTables:

    DNS_ONLY = {"queries": [{"failure": "dns_nxdomain_error", "query_type": "A", "hostname": "blocked.example", "engine": "system", "answers": []}]}
    RICH = {"queries": [
                {"failure": None, "query_type": "A", "hostname": "e.org", "engine": "system",
                 "answers": [{"answer_type": "A", "ipv4": "1.1.1.1"}, {"answer_type": "CNAME", "hostname": "cdn.e.org"}, {"answer_type": "AAAA", "ipv6": "::1"}]},
                {"failure": None, "query_type": "AAAA", "hostname": "e.org", "engine": "system", "answers": [{"answer_type": "AAAA", "ipv6": "2001:db8::1"}]}],
            "tcp_connect": [{"ip": "1.1.1.1", "port": 443, "status": {"success": True, "failure": None}},
                            {"ip": "2.2.2.2", "port": 80, "status": {"success": False, "failure": "connection_reset", "blocked": True}}]}

    def test_dns_censorship_survives_and_no_tcp_rows(self, spark):
        out = run(spark, [row(report_id="DNS", tk=self.DNS_ONLY)])
        assert out["parent"].count() == 1                                  # parent never filtered by empty child arrays
        d = out["dns"].collect()
        assert len(d) == 1 and d[0]["is_dns_failure"] is True and d[0]["total_answers_count"] == 0
        assert out["tcp"].count() == 0

    def test_no_cartesian_explosion_and_exact_counts(self, spark):
        out = run(spark, [row(report_id="RICH", tk=self.RICH)])
        assert out["parent"].count() == 1
        assert out["dns"].count() == 2 and out["tcp"].count() == 2            # 2 + 2, not 2 x 2
        first = out["dns"].orderBy("query_pos").first()
        assert (first["total_answers_count"], first["ipv4_answers_count"], first["cname_answers_count"]) == (3, 1, 1)
        t = {r["tcp_pos"]: r.asDict() for r in out["tcp"].collect()}
        assert t[1]["is_blocked"] is True and t[1]["failure_reason"] == "connection_reset" and t[0]["destination_port"] == 443

    def test_child_keys_unique_deterministic_and_repartition_stable(self, spark):
        rows = [row(report_id=f"K{i}", tk=self.RICH) for i in range(5)]
        a = run(spark, rows)
        b = silver.transform_bronze_to_silver(spark.createDataFrame(rows, BRONZE_SCHEMA).repartition(7), taxonomy(spark), PEPPER, "B2", LOAD_TS)
        for key, col in (("dns", "query_id"), ("tcp", "tcp_connect_id"), ("app", "app_measurement_id")):
            ids_a = sorted(r[col] for r in a[key].collect())
            assert len(ids_a) == len(set(ids_a))
            assert ids_a == sorted(r[col] for r in b[key].collect())

    def test_no_null_only_child_rows(self, spark):
        out = run(spark, [row(report_id="E1", tk={"queries": [], "tcp_connect": []}), row(report_id="E2", tk={})])
        assert out["dns"].count() == 0 and out["tcp"].count() == 0

    def test_app_routing_and_parent_consistency(self, spark):
        rows = [row(report_id="WA", test="whatsapp", inp=None, tk={"whatsapp_web_status": "blocked", "whatsapp_endpoints_status": "ok"}),
                row(report_id="TG", test="telegram", inp=None, tk={"telegram_http_blocking": True, "telegram_tcp_blocking": False}),
                row(report_id="SG", test="signal", inp=None, tk={"signal_backend_status": "ok"}),
                row(report_id="TO", test="tor", inp=None, tk={"dir_port_accessible": 6, "dir_port_total": 10, "obfs4_accessible": 3, "obfs4_total": 14}),
                row(report_id="PS", test="psiphon", inp=None, tk={"bootstrap_time": 8.5, "failure": None}),
                row(report_id="FB", test="facebook_messenger", inp=None, tk={"facebook_tcp_blocking": False}),
                row(report_id="WEB")]
        out = run(spark, rows)
        app = {r["app_name"]: r.asDict() for r in out["app"].collect()}
        assert set(app) == {"whatsapp", "telegram", "signal", "tor", "psiphon", "facebook_messenger"}
        assert out["parent"].count() == 7                                      # app tests ALSO stay in the parent (1:1 grain)
        assert app["whatsapp"]["whatsapp_web_status"] == "blocked" and app["telegram"]["telegram_http_blocked"] is True
        assert (app["tor"]["tor_dir_port_accessible"], app["tor"]["tor_obfs4_total"]) == (6, 14)
        assert app["psiphon"]["psiphon_bootstrap_time"] == 8.5
        parent_ids = {r["measurement_id"]: r["is_anomaly"] for r in out["parent"].collect()}
        for r in out["app"].collect():
            assert parent_ids[r["app_measurement_id"]] == r["is_anomaly"]

    def test_real_sample_files_end_to_end(self, spark):
        recs = []
        for fn in ("sample_full_load_ooni.json", "sample_incremental_ooni.json"):
            for x in json.load(open(os.path.join(ROOT, "data", "samples", fn))):
                tk = x["test_keys"] if isinstance(x["test_keys"], dict) else json.loads(x["test_keys"])
                recs.append(row(uid=None, report_id=x["report_id"], inp=x.get("input"), test=x["test_name"],
                                start=x["measurement_start_time"], asn=x["probe_asn"], cc=x["probe_cc"], pip=x["probe_ip"],
                                resolver_ip=x["resolver_ip"], rasn=x["resolver_asn"], runtime=x["test_runtime"], tk=tk))
        out = run(spark, recs)
        assert out["invalid"].count() == 0 and out["parent"].count() == len(recs)
        assert out["parent"].select("measurement_id").distinct().count() == len(recs)
        assert out["dns"].count() > 0 and out["tcp"].count() > 0 and out["app"].count() == 6
        vectors = {r["tampering_vector"] for r in out["parent"].select("tampering_vector").distinct().collect()}
        assert vectors <= set(silver.TAMPERING_VECTORS) and "BENIGN" in vectors

## Delta integration (skipped without Delta JARs)

In [0]:
def _delta_session():
    try:
        from delta import configure_spark_with_delta_pip
        b = (SparkSession.builder.master("local[2]").appName("silver-delta")
             .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
             .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
             .config("spark.sql.shuffle.partitions", "2").config("spark.ui.enabled", "false"))
        s = configure_spark_with_delta_pip(b).getOrCreate()
        s.range(1).write.format("delta").mode("overwrite").save("/tmp/_silver_delta_probe")
        return s
    except Exception:
        return None


@pytest.fixture(scope="module")
def dspark():
    s = _delta_session()
    if s is None:
        pytest.skip("Delta Lake JARs unavailable in this environment")
    yield s
    s.stop()

In [0]:
class TestSilverDeltaIntegration:

    def test_merge_idempotency_zero_write_amplification(self, dspark, tmp_path):
        path = str(tmp_path / "silver")
        out = run(dspark, [row(report_id=f"I{i}", inp=f"https://s{i}.org/") for i in range(5)])
        m1 = silver.merge_silver_network_measurements(dspark, out["parent"], path, "2026-09-25", "2026-09-28")
        m2 = silver.merge_silver_network_measurements(dspark, out["parent"], path, "2026-09-25", "2026-09-28")
        assert (m1["num_target_rows_inserted"], m1["num_target_rows_updated"]) == (5, 0)
        assert (m2["num_target_rows_inserted"], m2["num_target_rows_updated"]) == (0, 0)
        assert dspark.read.format("delta").load(path).groupBy("measurement_id").count().filter("count > 1").count() == 0

    def test_batch_id_alone_never_updates_but_taxonomy_change_does(self, dspark, tmp_path):
        path = str(tmp_path / "silver")
        rows = [row(report_id="A", inp="https://twitter.com/x"), row(report_id="B", inp="https://unrelated.example/")]
        first = silver.transform_bronze_to_silver(dspark.createDataFrame(rows, BRONZE_SCHEMA), taxonomy(dspark), PEPPER, "BATCH_1", LOAD_TS)
        silver.merge_silver_network_measurements(dspark, first["parent"], path, None, None)
        again = silver.transform_bronze_to_silver(dspark.createDataFrame(rows, BRONZE_SCHEMA), taxonomy(dspark), PEPPER, "BATCH_2", LOAD_TS)
        assert silver.merge_silver_network_measurements(dspark, again["parent"], path, None, None)["num_target_rows_updated"] == 0
        new_ref = dspark.createDataFrame([("https://twitter.com/", "ANON", "Circumvention", "GLOBAL", "commit-new")], REF_SCHEMA)
        changed = silver.transform_bronze_to_silver(dspark.createDataFrame(rows, BRONZE_SCHEMA), new_ref, PEPPER, "BATCH_3", LOAD_TS)
        m = silver.merge_silver_network_measurements(dspark, changed["parent"], path, None, None)
        assert m["num_target_rows_updated"] >= 1 and m["num_target_rows_inserted"] == 0

    def test_child_tables_idempotent_and_conform(self, dspark, tmp_path):
        out = run(dspark, [row(report_id="C", tk=TestSilverChildTables.RICH)])
        p = str(tmp_path / "dns")
        a = silver.merge_child_delta_table(dspark, out["dns"], p, silver.SILVER_DNS_SCHEMA, "query_id", None, None)
        b = silver.merge_child_delta_table(dspark, out["dns"], p, silver.SILVER_DNS_SCHEMA, "query_id", None, None)
        assert a["num_target_rows_inserted"] == 2 and b["num_target_rows_inserted"] == 0

## Run the tests

In [0]:
import ipytest
ipytest.autoconfig()
ipytest.run("-v", "-p", "no:cacheprovider")